In [50]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Read data

In [51]:
with open('../input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [52]:
print(f'Characters in text: {len(text)}')

Characters in text: 1115394


In [53]:
print(f'{text[:100]}')

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


# Getting unique characters and vocabulary size

In [54]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print("".join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


# Character level tokenizer

In [55]:
encoded_dict = {chars[i]: i for i in range(vocab_size)}
decoded_dict = {i: chars[i] for i in range(vocab_size)}

encode = lambda s: [encoded_dict[c] for c in s]
decode = lambda l: "".join([decoded_dict[i] for i in l])

print(encode("di si kompa"))
print(decode(encode("di si kompa")))

[42, 47, 1, 57, 47, 1, 49, 53, 51, 54, 39]
di si kompa


In [56]:
import torch

data = encode(text)
data = torch.tensor(data, dtype=torch.long)
print(data.shape)

torch.Size([1115394])


# Splitting train and val

In [57]:
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

In [58]:
block_size = 8
train_data[:block_size+1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [59]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f'when input is {context}, target is {target}')

when input is tensor([18]), target is 47
when input is tensor([18, 47]), target is 56
when input is tensor([18, 47, 56]), target is 57
when input is tensor([18, 47, 56, 57]), target is 58
when input is tensor([18, 47, 56, 57, 58]), target is 1
when input is tensor([18, 47, 56, 57, 58,  1]), target is 15
when input is tensor([18, 47, 56, 57, 58,  1, 15]), target is 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]), target is 58


In [60]:
torch.manual_seed(1337)

batch_size = 4
block_size = 8

def get_batch(split: str):
    data = train_data if split == "train" else val_data
    idx = torch.randint(len(data) - block_size, size=(batch_size, ))
    x = torch.stack([data[i:i+block_size] for i in idx])
    y = torch.stack([data[i+1:i+block_size+1] for i in idx])
    return x, y

xb, yb = get_batch("train")
print(xb.shape)
print(xb, "\n--------------------------------------------")
print(yb.shape)
print(yb, "\n--------------------------------------------")

for b in range(batch_size):
    for t in range(block_size):
        context = xb[b, :t+1]
        target = yb[b, t]
        print(f'When context is {context}, target is {target}')

torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]]) 
--------------------------------------------
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]]) 
--------------------------------------------
When context is tensor([24]), target is 43
When context is tensor([24, 43]), target is 58
When context is tensor([24, 43, 58]), target is 5
When context is tensor([24, 43, 58,  5]), target is 57
When context is tensor([24, 43, 58,  5, 57]), target is 1
When context is tensor([24, 43, 58,  5, 57,  1]), target is 46
When context is tensor([24, 43, 58,  5, 57,  1, 46]), target is 43
When context is tensor([24, 43, 58,  5, 57,  1, 46, 43]), target is 39
When context is tensor([44]), target is 53
When context is tensor([44, 53]), t

# Simples language model, Bigram

In [61]:
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        # each token reads directly off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):

        # idx and targets are both (B, T) tensor of integers
        logits = self.token_embedding_table(idx) # (B, T, C)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.reshape(B*T, C)
            targets = targets.reshape(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss
    
    def generate(self, idx, max_new_tokens):
        
        for _ in range(max_new_tokens):
            # idx is (B,T) array of indices in the current context
            logits, loss = self(idx)
            # focus only on last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)

        return idx

   
model = BigramLanguageModel(vocab_size)
logits, loss = model(xb, yb)
print(logits.shape)
print(loss)

torch.Size([32, 65])
tensor(4.8786, grad_fn=<NllLossBackward0>)


In [62]:
print(decode(model.generate(idx=torch.zeros((1,1), dtype=torch.long), max_new_tokens=100)[0].tolist()))


Sr?qP-QWktXoL&jLDJgOLVz'RIoDqHdhsV&vLLxatjscMpwLERSPyao.qfzs$Ys$zF-w,;eEkzxjgCKFChs!iWW.ObzDnxA Ms$3


In [63]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-2)

In [64]:
batch_size = 64

for steps in range(10000):

    xb, yb = get_batch("train")

    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())

2.416156053543091


In [68]:
print(decode(model.generate(idx=torch.zeros((1,1), dtype=torch.long), max_new_tokens=500)[0].tolist()))


bon's t bushe beg corallat:
Ty IAy arr engheeangde direl t,

Wishatan adgixcatitour knerethe isit:
OLAThe ciow my nt nllours hyofe lllavenel'dongharverd t ta gat ste imedumy teas, thiowait bulye tor ou weld tan

HArethe othims'l pertitond
FRTI:
Pere s wrsie ureend f ade, I,
Cotemef!

BYorgry inotheant ce R:
TEPro:
ANTExcoth wairudort t whtheay the wousheryoloremestowized S:
Whe movathishib k by fe fof pacerill h olave a I mber.

HAncell, forate ontad gavisores; e hauindind pal wineatse.
NThoumel


# Mathematical trick in self attention

In [77]:
torch.manual_seed(1337)

B, T, C = 4, 8, 2
x = torch.randn(B, T, C)
x.shape

torch.Size([4, 8, 2])

# Basic example of averaging using for loops

- v1

In [92]:
xbow = torch.zeros((B,T,C))
for b in range(B):
    for t in range(T):
        xprev = x[b, :t+1] # up until t-th token, (t, C)
        xbow[b, t] = torch.mean(xprev, dim=0)

xbow[0]

tensor([[ 0.1808, -0.0700],
        [-0.0894, -0.4926],
        [ 0.1490, -0.3199],
        [ 0.3504, -0.2238],
        [ 0.3525,  0.0545],
        [ 0.0688, -0.0396],
        [ 0.0927, -0.0682],
        [-0.0341,  0.1332]])

# Better averaging using triangular matrix and torch sum

- v2

In [91]:
w = torch.tril(torch.ones((T, T)))
w = w / torch.sum(w, dim=1, keepdim=True)

xbow2 = w @ x

xbow2[0]

tensor([[ 0.1808, -0.0700],
        [-0.0894, -0.4926],
        [ 0.1490, -0.3199],
        [ 0.3504, -0.2238],
        [ 0.3525,  0.0545],
        [ 0.0688, -0.0396],
        [ 0.0927, -0.0682],
        [-0.0341,  0.1332]])

# Adding softmax to triangular matrix

In [90]:
tril = torch.tril(torch.ones((T, T)))
w = torch.zeros((T, T))
w = w.masked_fill(tril == 0, float("-inf"))
w = F.softmax(w, dim=-1)
xbow3 = w @ x

xbow3[0]

tensor([[ 0.1808, -0.0700],
        [-0.0894, -0.4926],
        [ 0.1490, -0.3199],
        [ 0.3504, -0.2238],
        [ 0.3525,  0.0545],
        [ 0.0688, -0.0396],
        [ 0.0927, -0.0682],
        [-0.0341,  0.1332]])